# End-to-end pipeline: NLLB baseline

In [1]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "preprocessing" / "align.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. Data

In [2]:
from src.data import TedData

ted = TedData()

## 2. Preprocessing

In [3]:
SRC_LANG, TGT_LANG = "en", "az"
DIRECTION = f"{SRC_LANG}-{TGT_LANG}"

train_pairs = [(s, t) for _, s, t in ted.pairs("train", SRC_LANG, TGT_LANG)][:1000]
dev_pairs = [(s, t) for _, s, t in ted.pairs("dev", SRC_LANG, TGT_LANG)][:100]
test_pairs = [(s, t) for _, s, t in ted.pairs("test", SRC_LANG, TGT_LANG)][:100]
len(train_pairs), len(dev_pairs), len(test_pairs)

(1000, 100, 100)

## 3. Model

In [4]:
from src.model import build_model, translate_corpus, save_translations

MODEL_NAME, SYSTEM, MAX_LENGTH = "nllb", "nllb", 128

model = build_model(MODEL_NAME, None, None, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model.fit(pairs=list(train_pairs))

## 4. Evaluation

In [5]:
from src.evaluation import evaluate_direction

hyps = translate_corpus(model, [s for s, _ in test_pairs], max_length=MAX_LENGTH)
out_dir = ROOT / "results" / "translations" / SYSTEM
save_translations(out_dir, DIRECTION, [s for s, _ in test_pairs], [t for _, t in test_pairs], hyps)

scores = {}
for metric in ("bleu", "chrf", "comet"):
    try:
        scores[metric] = evaluate_direction(out_dir, DIRECTION, metric)
    except Exception as e:
        scores[metric] = str(e)
scores

  0%|          | 0/100 [00:00<?, ?it/s]

  1%|          | 1/100 [00:05<08:56,  5.42s/it]

  2%|▏         | 2/100 [00:06<04:38,  2.84s/it]

  3%|▎         | 3/100 [00:07<03:30,  2.17s/it]

  4%|▍         | 4/100 [00:08<02:37,  1.64s/it]

...

 98%|█████████▊| 98/100 [02:33<00:02,  1.15s/it]

 99%|█████████▉| 99/100 [02:34<00:00,  1.01it/s]

100%|██████████| 100/100 [02:35<00:00,  1.09s/it]

100%|██████████| 100/100 [02:35<00:00,  1.56s/it]

{'bleu': {'direction': 'en-az',
  'metric': 'bleu',
  'n_sentences': 100,
  'score': 8.82897006969008,
  'signature': 'nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0'},
 'chrf': {'direction': 'en-az',
  'metric': 'chrf',
  'n_sentences': 100,
  'score': 37.39051920634665,
  'signature': 'nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0'},
 'comet': "cannot import name '_HashedSeq' from 'functools' (/Users/kknopp/.pyenv/versions/3.14.2/lib/python3.14/functools.py)"}